# Landslide Prediction Model
## File: models/train_landslide.ipynb

**Ye notebook kya karta hai:**
- Realistic synthetic dataset generate karta hai (Western Ghats + Himalayas based)
- Features engineer karta hai
- Random Forest model train karta hai
- Evaluation karta hai (F1, AUC, Confusion Matrix)
- `landslide_model.pkl` save karta hai

**Run karo:** Kernel → Restart & Run All

**Output:** `landslide_model.pkl` → Copy karo `backend/` folder mein

## Step 1 — Libraries Install & Import

In [1]:
# Agar install nahi hain toh:
# !pip install scikit-learn numpy pandas matplotlib seaborn

import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

print("Libraries loaded!")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Libraries loaded!
NumPy: 2.3.0
Pandas: 2.3.1


## Step 2 — Dataset Generate Karo

In [2]:
def generate_landslide_data(n=5000, seed=42):
    
    #Realistic landslide dataset for India.
    #Physics-based probability:
    #- High slope + wet soil + heavy rain = high risk
    #- Low NDVI (less vegetation) = more risk
    #- Near river = more risk
    
    #Real data sources for future:
    #- NASA POWER: https://power.larc.nasa.gov
    #- USGS Landslide DB: https://catalog.data.gov
    #- ISRO Bhuvan DEM: https://bhuvan.nrsc.gov.in
    
    np.random.seed(seed)
    
    # Geographic (India bounds)
    lat = np.random.uniform(10, 32, n)
    lon = np.random.uniform(74, 97, n)
    
    # Dates (2020-2025)
    dates = pd.date_range("2020-01-01", periods=n, freq="6h")
    month = dates.month
    
    # Terrain
    elevation  = np.random.uniform(200, 3500, n)
    slope      = np.random.uniform(5, 65, n)
    river_dist = np.random.exponential(3.0, n).clip(0.1, 30)
    ndvi_base  = np.random.uniform(0.1, 0.85, n)
    
    # Weather (monsoon pattern)
    monsoon_factor = 1 + 2.5 * np.exp(-((month - 7.5)**2) / 4)
    rainfall_mm    = np.abs(np.random.normal(30 * monsoon_factor, 25, n))
    past_rain_3day = rainfall_mm * np.random.uniform(1.5, 4.0, n)
    past_rain_7day = past_rain_3day * np.random.uniform(1.8, 4.5, n)
    
    # Soil & climate
    soil_moisture = (0.3 + 0.5*(past_rain_7day/past_rain_7day.max())
                     + np.random.normal(0, 0.05, n)).clip(0.1, 1.0)
    temperature   = np.random.uniform(10, 35, n)
    humidity      = (50 + 0.3*rainfall_mm + np.random.normal(0, 8, n)).clip(30, 100)
    ndvi          = (ndvi_base - 0.15*(rainfall_mm/rainfall_mm.max())
                     + np.random.normal(0, 0.05, n)).clip(0.05, 0.9)
    
    # Physics-based probability
    p  = 0.02
    p += 0.35 * (slope / 65)
    p += 0.25 * (rainfall_mm / rainfall_mm.max())
    p += 0.15 * (past_rain_7day / past_rain_7day.max())
    p += 0.15 * soil_moisture
    p += 0.10 * (1 / (river_dist + 0.5))
    p += 0.08 * (1 - ndvi)
    p += 0.05 * (elevation / 3500)
    p += np.random.normal(0, 0.04, n)
    p  = p.clip(0, 1)
    
    # ~15% positive rate (realistic)
    landslide = (p > np.percentile(p, 85)).astype(int)
    
    df = pd.DataFrame({
        'rainfall_mm':      rainfall_mm.round(2),
        'past_rain_3day':   past_rain_3day.round(2),
        'past_rain_7day':   past_rain_7day.round(2),
        'soil_moisture':    soil_moisture.round(4),
        'ndvi':             ndvi.round(4),
        'elevation':        elevation.round(1),
        'slope':            slope.round(2),
        'river_distance_km':river_dist.round(3),
        'temperature':      temperature.round(1),
        'humidity':         humidity.round(1),
        'month':            month,
        'landslide':        landslide
    })
    return df

df = generate_landslide_data()
df.to_csv('landslide_dataset.csv', index=False)
print(f"Dataset: {df.shape}")
print(f"Landslide rate: {df['landslide'].mean():.1%}")
df.head()

Dataset: (5000, 12)
Landslide rate: 15.0%


,rainfall_mm,past_rain_3day,past_rain_7day,soil_moisture,ndvi,elevation,slope,river_distance_km,temperature,humidity,month,landslide
0,14.23,51.05,184.80,0.3541,0.1492,1433.0,34.98,3.928,22.4,48.4,1,0
1,83.51,324.55,684.16,0.4022,0.1054,1298.6,49.80,0.612,31.8,72.8,1,1
2,71.64,130.63,587.81,0.4368,0.4604,781.3,38.76,1.277,32.3,83.2,1,0
3,48.28,171.76,549.76,0.3883,0.8243,2204.0,10.00,3.266,24.7,67.6,1,0
4,21.43,76.30,324.81,0.4913,0.4689,1772.9,16.13,1.974,17.0,61.7,1,0


## Step 3 — Feature Engineering

In [3]:
FEATURES = [
    'rainfall_mm','past_rain_3day','past_rain_7day',
    'soil_moisture','ndvi','elevation','slope',
    'river_distance_km','temperature','humidity',
    # Engineered features:
    'rain_intensity_ratio','wetness_index','slope_moisture_risk',
    'vegetation_risk','rain_stress','topo_risk',
    'heat_humidity','is_monsoon'
]

def engineer(df):
    df = df.copy()
    df['rain_intensity_ratio'] = df['rainfall_mm'] / (df['past_rain_7day'] + 1)
    df['wetness_index']        = df['soil_moisture'] * df['rainfall_mm']
    df['slope_moisture_risk']  = df['slope'] * df['soil_moisture']
    df['vegetation_risk']      = 1 - df['ndvi']
    df['rain_stress']          = df['past_rain_3day'] + df['past_rain_7day']
    df['topo_risk']            = df['slope'] / (df['river_distance_km'] + 0.1)
    df['heat_humidity']        = df['humidity'] * df['temperature'] / 100
    df['is_monsoon']           = df['month'].isin([6,7,8,9]).astype(int)
    return df

df_eng = engineer(df)
print(f"Total features: {len(FEATURES)}")
print("New features added:")
for f in FEATURES[10:]:
    print(f"  - {f}")

Total features: 18
New features added:
  - rain_intensity_ratio
  - wetness_index
  - slope_moisture_risk
  - vegetation_risk
  - rain_stress
  - topo_risk
  - heat_humidity
  - is_monsoon


## Step 4 — Train Model

In [4]:
X = df_eng[FEATURES]
y = df_eng['landslide']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Positive rate: {y_train.mean():.1%}")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print("Model trained!")

Train: 4000, Test: 1000
Positive rate: 15.0%
Model trained!


## Step 5 — Evaluation

In [5]:
pred  = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print("=" * 50)
print("  LANDSLIDE MODEL RESULTS")
print("=" * 50)
print(f"  F1-Score : {f1_score(y_test, pred):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, proba):.4f}")
print()
print(classification_report(y_test, pred,
      target_names=['No Landslide', 'Landslide']))

  LANDSLIDE MODEL RESULTS
  F1-Score : 0.7709
  ROC-AUC  : 0.9723

              precision    recall  f1-score   support

No Landslide       0.95      0.98      0.96       850
   Landslide       0.85      0.71      0.77       150

    accuracy                           0.94      1000
   macro avg       0.90      0.84      0.87      1000
weighted avg       0.93      0.94      0.93      1000



## Step 6 — Plots

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f172a')
for ax in axes:
    ax.set_facecolor('#1e293b')
    for s in ax.spines.values():
        s.set_edgecolor('#334155')

# Feature Importance
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
axes[0].barh(fi.index, fi.values, color='#f97316', edgecolor='none')
axes[0].set_title('Feature Importance', color='white', pad=10)
axes[0].tick_params(colors='white', labelsize=8)

# Confusion Matrix
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No Slide','Slide'],
            yticklabels=['No Slide','Slide'])
axes[1].set_title('Confusion Matrix', color='white', pad=10)
axes[1].tick_params(colors='white')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, proba)
axes[2].plot(fpr, tpr, color='#f97316', lw=2,
             label=f"AUC={roc_auc_score(y_test,proba):.3f}")
axes[2].plot([0,1],[0,1],'--',color='#475569')
axes[2].set_title('ROC Curve', color='white', pad=10)
axes[2].tick_params(colors='white')
axes[2].legend(facecolor='#1e293b', labelcolor='white')

plt.suptitle('Landslide Model Evaluation', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('landslide_evaluation.png', dpi=130,
            bbox_inches='tight', facecolor='#0f172a')
plt.show()
print("Plot saved!")

Plot saved!


## Step 7 — Save Model

In [7]:
with open('landslide_model.pkl', 'wb') as f:
    pickle.dump({
        'model':    model,
        'features': FEATURES,
        'version':  '1.0',
        'disaster': 'landslide'
    }, f)

print("landslide_model.pkl saved!")
print()
print("NEXT STEP:")
print("  Copy 'landslide_model.pkl' to your 'backend/' folder")

landslide_model.pkl saved!

NEXT STEP:
  Copy 'landslide_model.pkl' to your 'backend/' folder
